# SwiGLU FFN - Fused Gate/Up Projection Triton Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qooba/bielik-anatomy-triton/blob/main/notebooks/ep06-swiglu.ipynb)

Part of the [Bielik Anatomy](https://github.com/qooba/bielik-anatomy-triton) series - Episode 6: Feed-Forward Network. Full write-up: [ep06-feed-forward-network.md](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep06-feed-forward-network.md).

Bielik's FFN uses a SwiGLU activation with separate gate and up projections. This notebook runs the fused Triton kernel (`kernels/ffn/swiglu_fused.py`) that computes both projections and the SwiGLU activation in a single kernel launch.

This notebook lets you run this kernel's correctness check and benchmark on a free Colab GPU - no local NVIDIA hardware required.

**Runtime:** select a GPU runtime before running: `Runtime -> Change runtime type -> T4 GPU` (free tier works fine).

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = "bielik-anatomy-triton"
if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/qooba/bielik-anatomy-triton {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 1. Correctness check

Compare the Triton kernel output against a plain PyTorch reference implementation.

In [ ]:
import torch
import torch.nn.functional as F
from kernels.ffn.swiglu_fused import fused_ffn_swiglu

torch.manual_seed(0)
seq_len, hidden, inter = 128, 1536, 8960
x = torch.randn(seq_len, hidden, device="cuda", dtype=torch.float16)
W_gate = torch.randn(hidden, inter, device="cuda", dtype=torch.float16) * 0.02
W_up = torch.randn(hidden, inter, device="cuda", dtype=torch.float16) * 0.02

def swiglu_ref(x, W_gate, W_up):
    gate = x @ W_gate
    up = x @ W_up
    return F.silu(gate) * up

out_triton = fused_ffn_swiglu(x, W_gate, W_up)
out_ref = swiglu_ref(x, W_gate, W_up)

max_diff = (out_triton - out_ref).abs().max().item()
print(f"max abs diff vs PyTorch reference: {max_diff:.6f}")
assert torch.allclose(out_triton, out_ref, atol=2e-2, rtol=2e-2)
print("Correctness check passed.")

## 2. Benchmark

Runs the same sweep as `make benchmark-swiglu` and regenerates the plots used in the episode doc. Takes roughly 1-3 minutes on a T4.

> **Note on T4 (Colab free tier):** T4 is Turing (compute capability 7.5), which has no native BF16 tensor cores — those arrived with Ampere (RTX 30-series, A100, and up). This kernel benchmarks BF16 `tl.dot` matmuls, so on T4 both the Triton and PyTorch BF16 providers fall back to a slow, non-tensor-core path. The TFLOPS numbers below will look much worse than the docs (measured on an RTX 4060 Ti, Ada) and are **not representative of the kernel's real performance** — for numbers that match the docs, switch to an A100/L4 runtime (Colab Pro) via `Runtime -> Change runtime type`.

In [ ]:
!PYTHONPATH=. python benchmarks/ffn/benchmark_swiglu_fused.py --save-plots --plot-dir=docs/plots/ffn/

## 3. Results

In [ ]:
from IPython.display import Image, display

plot_dir = "docs/plots/ffn/"
for fname in [
    "swiglu-ffn-tflops-vs-inter-size.png",
    "swiglu-ffn-tflops-vs-seq-len.png",
]:
    display(Image(filename=f"{plot_dir}{fname}"))

---
[Full episode write-up](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep06-feed-forward-network.md) · [Series overview](https://github.com/qooba/bielik-anatomy-triton/blob/main/README.md)